# 03 - Diagnostics, Regularization, and Interpretability

Executed notebook for regression diagnostics and regularization. It includes residual checks, Breusch-Pagan, Jarque-Bera, VIF, Ridge/Lasso comparison, coefficient shrinkage, and interpretation guidance.

## Learning objectives

Students will learn to inspect assumptions, quantify multicollinearity, compare regularized models, and explain when a regression model is useful for prediction versus inference.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
DATA_DIR = Path.cwd().parent / 'data' if (Path.cwd().parent / 'data').exists() else Path.cwd() / 'data'
print('Libraries imported.')
print('Dataset path resolved.')

Libraries imported.
Dataset path resolved.


## 1. Load data

In [2]:
df = pd.read_csv(DATA_DIR / 'multiple_regression_marketing_sales.csv')
df.head()

  campaign_id  tv_spend_k  radio_spend_k  social_spend_k  price_index  \
0        C001      146.00          64.71           18.72       116.35   
1        C002       22.08          73.15           36.77       118.69   
2        C003      142.26          65.27           28.03        93.91   
3        C004       54.75          36.33           13.81       116.59   
4        C005       64.26          44.41            9.60       108.44   

   competitor_spend_k season  sales_k_units  
0              164.96     Q2          67.05  
1               71.61     Q3          23.68  
2              138.89     Q1          85.06  
3               27.86     Q1           9.51  
4               77.28     Q4          24.60  

## 2. Fit OLS model

We use statsmodels because diagnostics and inference are easier to access.

In [3]:
ols = smf.ols('sales_k_units ~ tv_spend_k + radio_spend_k + social_spend_k + price_index + competitor_spend_k + C(season)', data=df).fit()
print(f'R2          : {ols.rsquared:.4f}')
print(f'Adjusted R2: {ols.rsquared_adj:.4f}')
print(f'AIC         : {ols.aic:.2f}')
print(f'BIC         : {ols.bic:.2f}')

R2          : 0.9657
Adjusted R2: 0.9633
AIC         : 875.63
BIC         : 900.72


## 3. Residual analysis

Residual plots help identify pattern, bias, heteroscedasticity, and outlier behavior.

In [4]:
residuals = ols.resid
fitted = ols.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(fitted, residuals, alpha=0.8)
axes[0].axhline(0, linewidth=1)
axes[0].set_title('OLS residuals vs fitted values')
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[1].hist(residuals, bins=15, edgecolor='black')
axes[1].set_title('OLS residual distribution')
plt.tight_layout()
plt.show()
print('Rendered residual-vs-fitted plot and residual histogram.')

Rendered residual-vs-fitted plot and residual histogram.


## 4. Diagnostic tests

| Test | Purpose |
|---|---|
| Breusch-Pagan | Checks heteroscedasticity |
| Jarque-Bera | Checks residual normality |

In [5]:
bp_stat, bp_p_value, _, _ = het_breuschpagan(residuals, ols.model.exog)
jb_result = stats.jarque_bera(residuals)
diagnostics = pd.DataFrame({'diagnostic': ['R2', 'Adjusted R2', 'Breusch-Pagan p-value', 'Jarque-Bera p-value', 'AIC', 'BIC'], 'value': [ols.rsquared, ols.rsquared_adj, bp_p_value, jb_result.pvalue, ols.aic, ols.bic]})
diagnostics

              diagnostic       value
0                     R2    0.965742
1            Adjusted R2    0.963273
2  Breusch-Pagan p-value    0.099542
3   Jarque-Bera p-value    0.000388
4                    AIC  875.628238
5                    BIC  900.715663

## 5. Multicollinearity with VIF

VIF above 5 deserves inspection; VIF above 10 is usually serious.

In [6]:
numeric_features = ['tv_spend_k', 'radio_spend_k', 'social_spend_k', 'price_index', 'competitor_spend_k']
X_vif = pd.get_dummies(df[numeric_features + ['season']], drop_first=True)
X_vif = sm.add_constant(X_vif).astype(float)
vif_table = pd.DataFrame({'feature': X_vif.columns, 'vif': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]})
vif_table.sort_values('vif', ascending=False).round(4)

                feature       vif
0                 const  124.3507
6             season_Q2    1.6849
8             season_Q4    1.6434
7             season_Q3    1.5670
4           price_index    1.1057
1            tv_spend_k    1.0755
3        social_spend_k    1.0654
5    competitor_spend_k    1.0572
2         radio_spend_k    1.0234

In [7]:
plot_vif = vif_table[vif_table['feature'] != 'const'].sort_values('vif')
plt.figure(figsize=(8,5))
plt.barh(plot_vif['feature'], plot_vif['vif'])
plt.axvline(5, linewidth=1)
plt.title('Variance Inflation Factor by feature')
plt.xlabel('VIF')
plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
print('Rendered VIF bar chart excluding intercept.')

Rendered VIF bar chart excluding intercept.


## 6. Regularization

Ridge adds an L2 penalty; Lasso adds an L1 penalty. Both are commonly used to improve model stability.

In [8]:
target = 'sales_k_units'
categorical_features = ['season']
X = df[numeric_features + categorical_features]
y = df[target]
preprocess_scaled = ColumnTransformer([('categorical', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features), ('numeric', StandardScaler(), numeric_features)])
print('Prepared scaled preprocessing pipeline for regularized models.')

Prepared scaled preprocessing pipeline for regularized models.


## 7. Compare Linear Regression, Ridge, and Lasso

In [9]:
models = {'LinearRegression': LinearRegression(), 'Ridge_alpha_1': Ridge(alpha=1.0), 'Ridge_alpha_10': Ridge(alpha=10.0), 'Lasso_alpha_0_05': Lasso(alpha=0.05, random_state=43, max_iter=10000), 'Lasso_alpha_0_5': Lasso(alpha=0.5, random_state=43, max_iter=10000)}
cv = KFold(n_splits=5, shuffle=True, random_state=43)
rows = []
for name, estimator in models.items():
    pipeline = Pipeline([('preprocess', preprocess_scaled), ('model', estimator)])
    scores = cross_validate(pipeline, X, y, cv=cv, scoring={'r2': 'r2', 'mae': 'neg_mean_absolute_error', 'rmse': 'neg_root_mean_squared_error'})
    rows.append({'model': name, 'mean_cv_r2': scores['test_r2'].mean(), 'mean_cv_mae': -scores['test_mae'].mean(), 'mean_cv_rmse': -scores['test_rmse'].mean()})
comparison = pd.DataFrame(rows).sort_values('mean_cv_rmse')
comparison.round(4)

                model  mean_cv_r2  mean_cv_mae  mean_cv_rmse
0    LinearRegression      0.9580       7.0894        9.3841
1  Lasso_alpha_0_05      0.9578       7.1475        9.4097
2      Ridge_alpha_1      0.9579       7.2091        9.4105
3    Lasso_alpha_0_5      0.9495       8.1726       10.2990
4     Ridge_alpha_10      0.9455       8.6992       10.7627

In [10]:
plot_data = comparison.sort_values('mean_cv_rmse', ascending=True)
plt.figure(figsize=(8,5))
plt.barh(plot_data['model'], plot_data['mean_cv_rmse'])
plt.title('Model comparison by cross-validated RMSE')
plt.xlabel('Mean CV RMSE')
plt.ylabel('Model')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
print('Rendered model comparison chart.')

Rendered model comparison chart.


## 8. Coefficient shrinkage demonstration

In [11]:
coef_rows = []
for name in ['LinearRegression', 'Ridge_alpha_10', 'Lasso_alpha_0_5']:
    estimator = models[name]
    pipeline = Pipeline([('preprocess', preprocess_scaled), ('model', estimator)])
    pipeline.fit(X, y)
    feature_names = pipeline.named_steps['preprocess'].get_feature_names_out()
    coefficients = pipeline.named_steps['model'].coef_
    for feature, coefficient in zip(feature_names, coefficients):
        coef_rows.append({'model': name, 'feature': feature, 'coefficient': coefficient})
coef_compare = pd.DataFrame(coef_rows)
coef_compare.head(10)

              model                feature  coefficient
0  LinearRegression  categorical__season_Q2     9.473192
1  LinearRegression  categorical__season_Q3     9.514449
2  LinearRegression  categorical__season_Q4    16.074842
3  LinearRegression     numeric__tv_spend_k    39.328985
4  LinearRegression  numeric__radio_spend_k    17.705972
5  LinearRegression numeric__social_spend_k     6.551248
6  LinearRegression   numeric__price_index    -8.616702
7  LinearRegression numeric__competitor_spend_k -7.115733
8    Ridge_alpha_10  categorical__season_Q2     8.111457
9    Ridge_alpha_10  categorical__season_Q3     8.159882

In [12]:
pivot = coef_compare.pivot(index='feature', columns='model', values='coefficient')
pivot.plot(kind='barh', figsize=(10,7))
plt.axvline(0, linewidth=1)
plt.title('Coefficient comparison: OLS vs Ridge vs Lasso')
plt.xlabel('Coefficient value')
plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()
print('Rendered coefficient shrinkage chart.')

Rendered coefficient shrinkage chart.


## Final diagnostic interpretation

- VIF values for real predictors are low, so multicollinearity is not a serious concern in this synthetic dataset.
- Breusch-Pagan p-value is above 0.05, so there is no strong heteroscedasticity signal.
- Jarque-Bera p-value is small, so residual normality should be discussed carefully for inference.
- Linear Regression has the best CV RMSE here, while Ridge and Lasso are included to teach regularization and coefficient shrinkage.

## Student exercises

1. Increase Ridge alpha and observe coefficient shrinkage.
2. Increase Lasso alpha and identify coefficients that become zero.
3. Add a Q-Q plot.
4. Remove `season` and rerun VIF.
5. Write a final governance note: prediction use, interpretation risk, and limitations.